In [1]:
import pandas as pd
from FunctionalCalculator import FunctionalScoreCalculator

# 👇 use ONE actual mapped file
file_path = "/home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1_merged_mapped.csv"

df = pd.read_csv(file_path)

calc = FunctionalScoreCalculator()

wt = calc.compute_wt(df)

print("\nWT RESULT:")
print(wt)

INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections



WT RESULT:
                                      selection_name  pre_count_wt  \
0  A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-0...       2176742   

   post_count_wt  
0        3365303  


In [2]:

import os
import pandas as pd

from FunctionalCalculator import FunctionalScoreCalculator


# =========================
# PATHS
# =========================
mapped_folder = (
    "/home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/"
    "results/func_scores/merged_output_clean/"
    "mapped_output_clean"
)

selection_file = (
    "/home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/"
    "results/func_scores/functional_selections_clean.csv"
)

output_folder = os.path.join(
    mapped_folder,
    "pipeline_output"
)

os.makedirs(output_folder, exist_ok=True)


# =========================
# LOAD METADATA
# =========================
selections = pd.read_csv(selection_file)

meta = selections[
    [
        "selection_name",
        "preselection_sample",
        "postselection_sample"
    ]
].rename(columns={
    "preselection_sample": "pre_sample",
    "postselection_sample": "post_sample"
})


# =========================
# INIT CALCULATOR
# =========================
calc = FunctionalScoreCalculator(
    pseudocount=0.5,
    min_preselection_counts=20,
    min_preselection_frac=1e-6
)


# =========================
# GET INPUT FILES
# =========================
files = [
    f for f in os.listdir(mapped_folder)
    if f.endswith(".csv")
    and "_pipeline" not in f
]

print(f"Found {len(files)} mapped files\n")


# =========================
# PROCESS FILES
# =========================
for file in files:

    filepath = os.path.join(
        mapped_folder,
        file
    )

    print(f"Processing: {file}")

    # -------------------------
    # Load mapped CSV
    # -------------------------
    df = pd.read_csv(filepath)

    # -------------------------
    # WT DEBUG
    # -------------------------
    wt_rows = df[
        df["n_codon_substitutions"] == 0
    ]

    print("\nWT DEBUG:")
    print(
        f"WT rows found: {len(wt_rows)}"
    )

    print(
        wt_rows[
            [
                "barcode",
                "pre_count",
                "post_count"
            ]
        ].head(10)
    )

    print(
        "\nWT TOTAL COUNTS:"
    )

    print(
        "Total WT pre_count:",
        wt_rows["pre_count"].sum()
    )

    print(
        "Total WT post_count:",
        wt_rows["post_count"].sum()
    )

    # -------------------------
    # Run calculator
    # -------------------------
    try:

        result = calc.run(df)

        # -------------------------
        # Merge metadata
        # -------------------------
        result = result.merge(
            meta,
            on="selection_name",
            how="left"
        )

        # Validate merge
        if result["pre_sample"].isna().any():

            raise ValueError(
                f"Metadata merge failed "
                f"for file: {file}"
            )

        # -------------------------
        # Rename columns
        # -------------------------
        result = result.rename(columns={
            "aa_substitutions_sequence":
            "aa_substitutions_sequential"
        })

        # -------------------------
        # Add library column
        # -------------------------
        result["library"] = (
            file.split("_")[0]
        )

        # -------------------------
        # Final column order
        # -------------------------
        final_columns = [
            "library",
            "pre_sample",
            "post_sample",
            "barcode",
            "func_score",
            "func_score_var",
            "pre_count",
            "post_count",
            "pre_count_wt",
            "post_count_wt",
            "pseudocount",
            "n_codon_substitutions",
            "aa_substitutions_sequential",
            "n_aa_substitutions",
            "aa_substitutions_reference",
            "pre_count_threshold"
        ]

        # Validate final columns
        missing_cols = [
            c for c in final_columns
            if c not in result.columns
        ]

        if missing_cols:

            raise ValueError(
                f"Missing final columns: "
                f"{missing_cols}"
            )

        result = result[
            final_columns
        ]

        # -------------------------
        # Save output
        # -------------------------
        output_path = os.path.join(
            output_folder,
            file.replace(
                ".csv",
                "_pipeline.csv"
            )
        )

        result.to_csv(
            output_path,
            index=False
        )

        print(
            f"\nSaved: {output_path}\n"
        )

    except Exception as e:

        print(
            f"\n❌ Error processing "
            f"{file}: {e}\n"
        )


print("ALL FILES PROCESSED SUCCESSFULLY")

Found 10 mapped files

Processing: A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1_merged_mapped.csv


INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 25
INFO: Computing functional scores...
INFO: Functional score calculation complete



WT DEBUG:
WT rows found: 3990
              barcode  pre_count  post_count
11   AAAAAAAAGAATTTAG        142           0
22   AAAAAAACATAGATAT        208         236
30   AAAAAAAGAATCTAAA        356        1113
60   AAAAAACCGTGTTAGC         57          93
92   AAAAAATAAACTGAAC        713         140
97   AAAAAATACAATCCAC         97         154
101  AAAAAATATAAATCGG        367         402
102  AAAAAATATGGAAACG        164         214
115  AAAAAATTAAAGAGAG        601         846
121  AAAAAATTCAGATCAA        226         318

WT TOTAL COUNTS:
Total WT pre_count: 2176742
Total WT post_count: 3365303

Saved: /home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/pipeline_output/A_2022-07-20_rescue-2_VSVG_control_1_vs_2022-07-20_rescue-2_no-antibody_control_1_merged_mapped_pipeline.csv

Processing: B_2022-08-04_rescue-2_VSVG_control_2_vs_2022-08-04_rescue-2_no-antibody_control_2_merged_mapped.csv


INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3024
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 33
INFO: Computing functional scores...
INFO: Functional score calculation complete



WT DEBUG:
WT rows found: 3024
              barcode  pre_count  post_count
29   AAAAAAAGGTCTGGCG        719        2272
81   AAAAAATCATTTCCAG       1660        4754
87   AAAAAATGCAACTGAA         82           0
93   AAAAACAAATAAAATA        198         357
99   AAAAACACCCCGAAAG        343         558
134  AAAAACTTCAAAACAA        553          61
153  AAAAAGATTCACCTAT        328         126
176  AAAAAGTTACAAGCTC        352         312
190  AAAAATACCACATAGT          0           0
192  AAAAATACGACCCAAT       6069       16049

WT TOTAL COUNTS:
Total WT pre_count: 2132928
Total WT post_count: 4100375

Saved: /home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/pipeline_output/B_2022-08-04_rescue-2_VSVG_control_2_vs_2022-08-04_rescue-2_no-antibody_control_2_merged_mapped_pipeline.csv

Processing: A_2022-09-01_rescue-3_VSVG_control_1_vs_2022-09-01_rescue-3_no-antibody_control_1_merged_mapped.csv


...


WT DEBUG:
WT rows found: 3024
              barcode  pre_count  post_count
29   AAAAAAAGGTCTGGCG        837        2195
81   AAAAAATCATTTCCAG       1679        3668
87   AAAAAATGCAACTGAA         58           0
93   AAAAACAAATAAAATA        169         195
99   AAAAACACCCCGAAAG        347         728
134  AAAAACTTCAAAACAA        485          90
153  AAAAAGATTCACCTAT        317          95
176  AAAAAGTTACAAGCTC        328         459
190  AAAAATACCACATAGT          0           0
192  AAAAATACGACCCAAT       5874       16261

WT TOTAL COUNTS:
Total WT pre_count: 2100524
Total WT post_count: 4445211

Saved: /home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/pipeline_output/B_2022-08-04_rescue-2_VSVG_control_1_vs_2022-08-04_rescue-2_no-antibody_control_1_merged_mapped_pipeline.csv

Processing: A_2022-09-27_rescue-3_VSVG_control_1_vs_2022-09-27_rescue-3_no-antibody_control_1_merged_mapped.csv


INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 58
INFO: Computing functional scores...
INFO: Functional score calculation complete



WT DEBUG:
WT rows found: 3990
              barcode  pre_count  post_count
11   AAAAAAAAGAATTTAG        326          53
22   AAAAAAACATAGATAT        608        1781
30   AAAAAAAGAATCTAAA       1192        1683
60   AAAAAACCGTGTTAGC        125         370
92   AAAAAATAAACTGAAC       1500          72
97   AAAAAATACAATCCAC        149          81
101  AAAAAATATAAATCGG        869         393
102  AAAAAATATGGAAACG        332         602
115  AAAAAATTAAAGAGAG       1436        1449
121  AAAAAATTCAGATCAA        545        1030

WT TOTAL COUNTS:
Total WT pre_count: 5028353
Total WT post_count: 7232941

Saved: /home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/pipeline_output/A_2022-09-27_rescue-3_VSVG_control_1_vs_2022-09-27_rescue-3_no-antibody_control_1_merged_mapped_pipeline.csv

Processing: A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2_merged_mapped.csv


INFO: Starting Functional Score Calculation...
INFO: Computing WT counts...
INFO: WT rows found: 3990
INFO: WT computed for 1 selections
INFO: Dynamic pre-count threshold: 24
INFO: Computing functional scores...
INFO: Functional score calculation complete



WT DEBUG:
WT rows found: 3990
              barcode  pre_count  post_count
11   AAAAAAAAGAATTTAG         96           1
22   AAAAAAACATAGATAT        247         503
30   AAAAAAAGAATCTAAA        425        1189
60   AAAAAACCGTGTTAGC         55         138
92   AAAAAATAAACTGAAC        591           4
97   AAAAAATACAATCCAC         79         135
101  AAAAAATATAAATCGG        359         816
102  AAAAAATATGGAAACG        152         213
115  AAAAAATTAAAGAGAG        708        1237
121  AAAAAATTCAGATCAA        220         594

WT TOTAL COUNTS:
Total WT pre_count: 2037345
Total WT post_count: 4457874

Saved: /home/lechiffre/HIV_Envelope_BF520_DMS_CD4bs_sera/results/func_scores/merged_output_clean/mapped_output_clean/pipeline_output/A_2022-07-20_rescue-2_VSVG_control_2_vs_2022-07-20_rescue-2_no-antibody_control_2_merged_mapped_pipeline.csv

ALL FILES PROCESSED SUCCESSFULLY
